## Imports, loading data and methods from `features.py`

In [44]:
import pandas as pd
import numpy as np
from features import *
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

data = pd.read_csv("train.csv")

pool_qc(data)
misc_feature(data)
alley(data)
fence(data)
mas_vnr_type(data)
fireplace_qu(data)
lot_frontage(data)
garage_qual(data)
garage_type(data)
garage_finish(data)
garage_cond(data)
bsmt_cond(data)
bsmt_qual(data)
bsmt_exposure(data)
bsmt_fin_type1(data)
bsmt_fin_type2(data)
garage_yr_blt(data)
mas_vnr_area(data)
electrical(data)
data = delete_outliers(data)
clip_rare_values(data)

X = data.drop(columns=["SalePrice", "Id"])
y = np.log(data["SalePrice"])

## Categorical and Numerical Features

In [57]:
cat_cols = X.select_dtypes(include=["string"]).columns
num_cols = X.select_dtypes(include=["int", "float"]).columns

## Categorical pipeline and Numerical pipeline + OneHotEncoder, StandardScaler, SimpleImputer

In [58]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown="ignore")),
])

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler()),
])

## ColumnTransformer

In [59]:
preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num', num_pipeline, num_cols),
])

## LinearRegression Baseline pipeline

In [48]:
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('linear', LinearRegression()),
])

scores = cross_val_score(baseline_pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
rmse = -scores.mean()
print(f"Baseline result: {rmse}")

Baseline result: 0.12421373645732689


## Baseline - LinearRegression

 - RMSE after imputation of missing values (log `SalePrice`, 5-fold CV): 0.1478
 - RMSE after 1st FeatureEngineering (Numerical Columns) (log `SalePrice`, 5-fold CV)L 0.1242

### Ridge/Lasso vs DecisionTreeRegressor vs RandomForestRegressor vs XGBRegressor

In [10]:
models = [
    ('Ridge', Ridge(random_state=42)),
    ('Lasso', Lasso(random_state=42, alpha=0.001)),
    ('DecisionTreeRegressor', DecisionTreeRegressor(random_state=42)),
    ('RandomForestRegressor', RandomForestRegressor(random_state=42)),
    ('XGBRegressor', XGBRegressor(random_state=42)),
]

results = []

for name, model in models:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])

    scores = cross_val_score(pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
    rmse = -scores.mean()
    results.append({'model': name, 'rmse': rmse})

df_results = pd.DataFrame(results)
df_results.head()

,model,rmse
0,Ridge,0.119441
1,Lasso,0.112691
2,DecisionTreeRegressor,0.200584
3,RandomForestRegressor,0.138364
4,XGBRegressor,0.135935


### Lasso GridSearch

In [18]:
lasso_param_grid = {
    'lasso__alpha': [0.0001, 0.001, 0.0012, 0.0015],
    'lasso__max_iter': [3000, 5000]
}

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lasso', Lasso(random_state=42)),
])

lasso_grid = GridSearchCV(pipeline, lasso_param_grid, cv=5, scoring='neg_root_mean_squared_error')
lasso_grid.fit(X, y)
print(lasso_grid.best_params_)
print(-lasso_grid.best_score_)

{'lasso__alpha': 0.001, 'lasso__max_iter': 3000}
0.11269130218671833


### Ridge GridSearch

In [20]:
ridge_param_grid = {
    'ridge__alpha': [0.1, 0.2, 1, 5],
}

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('ridge', Ridge(random_state=42)),
])

ridge_grid = GridSearchCV(pipeline, ridge_param_grid, cv=5, scoring='neg_root_mean_squared_error')
ridge_grid.fit(X, y)
print(ridge_grid.best_params_)
print(-ridge_grid.best_score_)

{'ridge__alpha': 5}
0.11564093373794133


### RandomForestRegressor GridSearch

In [30]:
rf_param_grid = {
    'rf__n_estimators': [200, 400],
    'rf__max_depth': [6, 10, 15, None],
    'rf__min_samples_split': [2, 3],
    'rf__min_samples_leaf': [1, 2, 3]
}

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42)),
])

rf_grid = GridSearchCV(pipeline, rf_param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
rf_grid.fit(X, y)
print(rf_grid.best_params_)
print(-rf_grid.best_score_)

{'rf__max_depth': None, 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 2, 'rf__n_estimators': 400}
0.13675463890475575


### XGBRegressor GridSearch

In [29]:
xgb_param_grid = {
    'xgb__n_estimators': [100, 200, 300, 400, 500],
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'xgb__max_depth': [2, 3, 4]
}

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', XGBRegressor(random_state=42)),
])

xgb_grid = GridSearchCV(pipeline, xgb_param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
xgb_grid.fit(X, y)
print(xgb_grid.best_params_)
print(-xgb_grid.best_score_)

{'xgb__learning_rate': 0.1, 'xgb__max_depth': 3, 'xgb__n_estimators': 400}
0.11736498703920373


### Final model (Lasso) and first submission

In [62]:
clean_params = {k.replace('lasso__', ''): v for k, v in lasso_grid.best_params_.items()}

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('lasso', Lasso(**clean_params, random_state=42)),
])

final_model.fit(X, y)

test_data = pd.read_csv('test.csv')
pool_qc(test_data)
misc_feature(test_data)
alley(test_data)
fence(test_data)
mas_vnr_type(test_data)
fireplace_qu(test_data)
lot_frontage(test_data)
garage_qual(test_data)
garage_type(test_data)
garage_finish(test_data)
garage_cond(test_data)
bsmt_cond(test_data)
bsmt_qual(test_data)
bsmt_exposure(test_data)
bsmt_fin_type1(test_data)
bsmt_fin_type2(test_data)
garage_yr_blt(test_data)
mas_vnr_area(test_data)
electrical(test_data)
clip_rare_values(test_data)

house_id = test_data['Id']
test_data = test_data.drop(columns='Id')
prediction = np.exp(final_model.predict(test_data))

submission = {
    'Id': house_id,
    'SalePrice': prediction,
}

submission_df = pd.DataFrame(submission)
submission_df.to_csv('submission.csv', index=False)